In [ ]:
%load_ext autoreload
%autoreload 2

from app.logger import *
logger = create_logger(name="temp_logger",log_dir="logs",redirect_stdout=False)

from pandas.errors import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import json, os, logging, warnings
from app.amc.fund_data import *
from app.utils import Helper

config = Helper.load_json("paths.json")
amc_path = config['amc_path']
mutual_fund = Helper.get_amc_paths(amc_path)

In [ ]:
# 1_0: Axis Mutual Fund
# 1_1: Axis Mutual Fund Passive
# 2_0: Baroda Bnp Paribas Mutual Fund
# 3_0: Aditya Birla Sun Life Mutual Fund
# 5_0: Bank Of India Mutual Fund
# 6_0: Canara Robeco Mutual Fund
# 7_0: Pgim India Mutual Fund
# 8_0: Dsp Mutual Fund
# 9_0: Edelweiss Mutual Fund
# 10_0: Quant Mutual Fund
# 11_0: Franklin Templeton Mutual Fund
# 12_1: Hdfc Mutual Fund Passive
# 12_0: Hdfc Mutual Fund
# 13_0: Hsbc Mutual Fund
# 14_0: Icici Prudential Mutual Fund
# 14_1: Icici Prudential Mutual Fund Passive
# 16_0: Bandhan Mutual Fund
# 18_0: 360 One Mutual Fund
# 20_0: Groww Mutual Fund
# 21_0: Invesco Mutual Fund
# 22_0: Jm Financial Mutual Fund
# 23_0: Kotak Mahindra Mutual Fund
# 25_0: Lic Mutual Fund
# 26_0: Mahindra Manulife Mutual Fund
# 27_0: Mirae Asset Mutual Fund
# 27_1: Mirae Asset Mutual Fund Passive
# 28_1: Motilal Oswal Mutual Fund
# 28_0: Motilal Oswal Mutual Fund
# 30_0: Ppfas Mutual Fund
# 32_0: Quantum Mutual Fund
# 33_0: Nippon India Mutual Fund
# 35_0: Sbi Mutual Fund
# 35_1: Sbi Mutual Fund Passive
# 36_0: Shriram Mutual Fund
# 37_0: Sundaram Mutual Fund
# 38_0: Tata Mutual Fund
# 39_0: Taurus Mutual Fund
# 40_0: Union Mutual Fund
# 41_0: Uti Mutual Fund
# 41_1: Uti Mutual Fund Passive
# 42_0: Whiteoak Mutual Fund
# 51_0: Iti Mutual Fund
# 55_0: Trust Mutual Fund
# 56_0: Navi Mutual Fund
# 56_1: Navi Mutual Fund Passive
# 57_0: Nj Mutual Fund
# 58_0: Samco Mutual Fund
# 59_0: Bajaj Finserv Mutual Fund
# 60_0: Helios Mutual Fund
# 71_0: Zerodha Mutual Fund
# 95_0: Old Bridge Mutual Fund
# 96_0: Angel One Mutual Fund
# 97_0: Unifi Mutual Fund
# 98_0: Jio BlackRock Mutual Fund

In [7]:
path = r"C:\Users\rando\Office Projects\mywork-repo\notebook\98_31-Aug-25_FS.pdf"
amc_id = "98_0"
amc_class = JioBlackRock(amc_id,path)

In [9]:
title,path_pdf = amc_class.check_and_highlight(path)
data = amc_class.get_data(path_pdf,title)
extracted_text = amc_class.get_generated_content(data)
final_text = amc_class.refine_extracted_data(extracted_text)
dfs = amc_class.merge_and_select_data(final_text)

In [10]:
save_path = os.path.join(amc_class.JSONPATH, amc_class.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
print(f"File Saved At: {save_path}")

# with open("data.json", 'w') as f:
#   json.dump(final_text, f, indent=2)
  
# with open("extract.json", 'w') as f:
#   json.dump(extracted_text, f, indent=2)

with open(save_path, 'r') as f:
  data = json.load(f)

    

File Saved At: C:\Users\rando\Office Projects\outputs\amc_output\json\98_31-Aug-25_FS.json


In [6]:
import os, json
from collections import defaultdict
import pandas as pd

static_keys = [
    "amc_name", "main_scheme_name", "mutual_fund_name", "benchmark_index", "monthly_aaum_date", 
    "monthly_aaum_value", "scheme_launch_date", "min_addl_amt", "min_addl_amt_multiple", 
    "min_amt", "min_amt_multiple",
]

load_keys = ["entry_load", "exit_load"]

metric_keys = [
    "alpha", "arithmetic_mean_ratio", "average_div_yield", "average_pb", "average_pe", "avg_maturity",
    "beta", "correlation_ratio", "downside_deviation", "information_ratio", "macaulay",
    "mod_duration", "port_turnover_ratio", "r_squared_ratio", "roe_ratio", "sharpe", "sortino_ratio",
    "std_dev", "tracking_error", "treynor_ratio", "upside_deviation", "ytm"
]

path = r"C:\Users\kaustubh.keny\Projects\JSON25\output\JSON FY 19\3"
xls_name = "FS_40_2020.xlsx"

def flatten_to_row(value, max_manager): 
    add_value = []
    for k in static_keys:
        val = value.get(k, "")
        if isinstance(val, list):
            val = ", ".join(val)
        add_value.append(val)

    metric_map = {m["name"]: m["value"] for m in value.get("metrics", [])}
    metric = [metric_map.get(k, "") for k in metric_keys]
    add_value.extend(metric)
    
    entry, exit = "", ""
    for l in value.get("load", []):
        if l.get("type") == "entry_load":
            entry = l.get("comment", "")
        elif l.get("type") == "exit_load":
            exit = l.get("comment", "")
    load = [entry, exit]
    add_value.extend(load)
    
    fms = value.get("fund_manager", [])
    for i in range(max_manager):
        if i < len(fms):
            fm = fms[i]
            add_value.append(fm.get("name", ""))
            add_value.append(fm.get("managing_fund_since", ""))
            add_value.append(fm.get("total_exp", ""))
            add_value.append(fm.get("qualification", ""))
        else:
            add_value.extend(["", "", "", ""])

    return add_value

file_paths = []
month_data = defaultdict(list)
sheet_columns = {}

for pth, _, files in os.walk(path):
    for f in files:
        file_path = os.path.join(pth, f)
        file_paths.append(file_path)

for paths in file_paths:
    with open(paths, "r", encoding="utf8") as f:
        df = json.load(f)

    sheet_name = df.get("metadata", {}).get("document_name", "")
    records = df.get("records", [])

    max_manager = 0
    for record in records:
        value = record.get("value", {})
        manager = value.get("fund_manager", [])
        if len(manager) > max_manager:
            max_manager = len(manager)

    print(f"{sheet_name} MAX MANAGER: {max_manager}")

    fund_manager_keys = []
    for i in range(max_manager):
        fund_manager_keys.extend([
            f"fund_manager_{i}",
            f"since_{i}",
            f"experience_{i}",
            f"qualification_{i}"
        ])

    final_keys = static_keys + metric_keys + load_keys + fund_manager_keys
    sheet_columns[sheet_name] = final_keys

    for record in records:
        value = record.get("value", {})
        row = flatten_to_row(value, max_manager)
        month_data[sheet_name].append(row)

month_data = dict(month_data)

with pd.ExcelWriter(xls_name) as writer:
    for month, rows in month_data.items():
        if not rows:
            print(f"No data for month: {month}")
            continue
        try:
            df = pd.DataFrame(rows, columns=sheet_columns[month])
            df.to_excel(writer, sheet_name=month[:31], index=False)
        except Exception as e:
            print(f"Month: {month} Error.")
            print(e)


3_28-Feb-19_FS.pdf MAX MANAGER: 3
3_30-Apr-19_FS.pdf MAX MANAGER: 3
3_30-Jun-19_FS.pdf MAX MANAGER: 3
3_30-Nov-19_FS.pdf MAX MANAGER: 3
3_30-Sep-19_FS.pdf MAX MANAGER: 3
3_31-Aug-19_FS.pdf MAX MANAGER: 3
3_31-Dec-19_FS.pdf MAX MANAGER: 3
3_31-Jan-19_FS.pdf MAX MANAGER: 3
3_31-Jul-19_FS.pdf MAX MANAGER: 3
3_31-Mar-19_FS.pdf MAX MANAGER: 3
3_31-May-19_FS.pdf MAX MANAGER: 3
3_31-Oct-19_FS.pdf MAX MANAGER: 3
